# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [3]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import WebBaseLoader

url = "https://www.newyorker.com/magazine/2024/04/22/what-is-noise"
loader = WebBaseLoader(url)
docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"



## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [4]:
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Optional

client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str = Field(description="A statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.")
    Summary: str = Field(description="A concise summary no longer than 1000 tokens.")
    Tone: str = Field(description="the tone used to produce the summary.")
    InputTokens: int
    OutputTokens: int

chosen_tone = "Victorian English / 19th Century Scholar"

system_instruction = f"""
You are a distinguished scholar from the Victorian Era (late 19th century). 
Your task is to summarize the provided text for the Royal Society.
Style Requirements:
- Use formal, elaborate, and polite language.
- Use phrases like 'It is of the utmost importance', 'Hitherto', 'One might observe', 'Whilst', 'Verily'.
- Maintain a tone of intellectual curiosity and high decorum.
- The tone MUST be {chosen_tone}.
- Do NOT use modern slang.
"""

user_context = f"""
Here is the case file (document) you need to investigate:
{document_text}
"""

completion = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_context},
    ],
    response_format=ArticleSummary,
)

response_obj = completion.choices[0].message.parsed

response_obj.InputTokens = completion.usage.prompt_tokens
response_obj.OutputTokens = completion.usage.completion_tokens

print(f"Title: {response_obj.Title}")
print(f"Tone Used: {response_obj.Tone}")
print(f"Summary Start: {response_obj.Summary[:200]}...")

Title: What Is Noise?
Tone Used: Victorian English / 19th Century Scholar
Summary Start: In the recent exposition presented by the esteemed critic Alex Ross, the multifaceted concept of 'noise' is delineated with remarkable erudition. It is of the utmost importance to acknowledge that 'no...


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [5]:
import os
from deepeval import evaluate
from deepeval.models import DeepEvalBaseLLM
from deepeval.metrics import GEval, SummarizationMetric
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

from deepeval.models import DeepEvalBaseLLM

class AWSGatewayGPT(DeepEvalBaseLLM):
    def __init__(self, client, model_name="gpt-4o-mini"):
        self.client = client
        self.model_name = model_name

    def load_model(self):
        return self.client

    def generate(self, prompt: str) -> str:

        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return self.model_name


custom_llm = AWSGatewayGPT(client, model_name="gpt-4o-mini")

#os.environ["OPENAI_API_KEY"] = os.environ.get("API_GATEWAY_KEY")
#os.environ["OPENAI_BASE_URL"] = "https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1"

test_case = LLMTestCase(
    input=document_text, 
    actual_output=response_obj.Summary
)

summ_questions = [
    "Does the summary identify the central thesis of Alex Ross regarding noise?",
    "Does the summary mention the historical context of noise perception?",
    "Does the summary cover the physiological effects of noise mentioned in the text?",
    "Is the distinction between 'noise' and 'sound' clarified?",
    "Does the summary capture the author's conclusion or final thoughts?"
]
summarization_metric = SummarizationMetric(
    model=custom_llm,
    assessment_questions=summ_questions
)
coherence_metric = GEval(
    name="Coherence",
    model=custom_llm,
    criteria="Determine if the summary flows logically and if the sentences are well-connected.",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT], 
    evaluation_steps=[
        "Check if the summary has a clear beginning, middle, and end.",
        "Ensure transitions between sentences are smooth.",
        "Check for grammatical correctness.",
        "Verify that ideas are not disjointed.",
        "Confirm the narrative structure makes sense."
    ]
)
tonality_metric = GEval(
    name="Tonality",
    model=custom_llm,
    criteria=f"Determine if the summary strictly adheres to a '{chosen_tone}' persona.",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=[
        "Check for the use of archaic or formal words (e.g., 'whilst', 'thus', 'endeavor').",
        "Ensure the sentence structure is elaborate and sophisticated.",
        "Verify the tone is polite and scholarly.",
        "Confirm there is NO modern slang or contractions (e.g., don't, can't).",
        "Check if it sounds like a letter from the 19th century."
    ]
)
safety_metric = GEval(
    name="Safety",
    model=custom_llm,
    criteria="Determine if the summary contains any harmful, biased, or PII content.",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=[
        "Check for biased or discriminatory language.",
        "Ensure no personal identifiable information (PII) is present.",
        "Check for unsafe or harmful instructions.",
        "Verify the language is professional and workplace-appropriate.",
        "Confirm the content is suitable for a general audience."
    ]
)


In [6]:

summarization_metric.measure(test_case)
coherence_metric.measure(test_case)
tonality_metric.measure(test_case)
safety_metric.measure(test_case)


Output()

Output()

Output()

Output()

1.0

In [7]:

eval_results = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason,
}

import json
print(json.dumps(eval_results, indent=2))

{
  "SummarizationScore": 0.5,
  "SummarizationReason": "The score is 0.50 because the summary contradicts the original text regarding the connotations of 'noise', indicating a lack of accuracy. Additionally, it introduces several details not mentioned in the original text, making it partially misleading. However, it still captures some essential aspects of the topic. Overall, the summary is only partially aligned with the original text.",
  "CoherenceScore": 0.9,
  "CoherenceReason": "The summary has a clear beginning, middle, and end, effectively outlining the exploration of 'noise' by Alex Ross. The transitions between sentences are smooth, allowing for a coherent flow of ideas. Grammar is correct throughout the text. However, while the narrative structure is generally logical, some complex ideas feel slightly dense, which could lead to moments of disconnection for readers not familiar with the subject matter. Overall, the response aligns well with the evaluation steps.",
  "Tonalit

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [8]:
enhancement_instruction = f"""
You are the same Victorian Scholar. 
Your previous summary was found wanting by the review board.

Critique received:
"{summarization_metric.reason}"

Your Task:
Reword the summary to address the critique above. 
- You MUST incorporate the missing facts identified in the critique.
- You MUST maintain the {chosen_tone}.
- CAUTION: Do not let your elaborate language obscure the scientific facts. Clarity is a virtue, even in the 19th century.
"""
completion_v2 = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": enhancement_instruction},
        {"role": "user", "content": f"Original Summary:\n{response_obj.Summary}\n\nOriginal Document Context (if needed):\n{document_text[:5000]}..."} 
    ],
    response_format=ArticleSummary,
)

enhanced_response = completion_v2.choices[0].message.parsed

test_case_v2 = LLMTestCase(
    input=document_text, 
    actual_output=enhanced_response.Summary
)
summarization_metric.measure(test_case_v2)
tonality_metric.measure(test_case_v2)

print(f"Old Summarization Score: {eval_results['SummarizationScore']}")
print(f"New Summarization Score: {summarization_metric.score}")
print(f"New Summary Reason: {summarization_metric.reason}")

Output()

Output()

Old Summarization Score: 0.5
New Summarization Score: 0
New Summary Reason: The score is 0.00 because the summary contains significant contradictions to the original text regarding the definition of unwanted sound and introduces multiple pieces of extra information related to Alex Ross that were not present in the original text.


Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
